# Phase 6 (post hoc): per-patient batch-normalization statistics at test time (phase6_patient_adaptation)

PREREGISTRATION §10 v9, P10. Adaptive batch normalization on the frozen checkpoints (final evaluation, v1 inputs; P7, v3
inputs): each record is classified by a copy of the network whose batch-norm statistics come from that record's beats (no
labels). Nothing is trained. CPU only.

| `PART` | inputs | what it does |
|---|---|---|
| `validation` | `ecgaug-v1-checkpoints`, `ecgaug-p7-runs` | the adoption rule on the validation records; writes `adoption.json` |
| `test` | `ecgaug-v1-checkpoints`, `ecgaug-p7-runs` | one posthoc DS2 read for the adopted pairs (sets `ALLOW_TEST_POSTHOC=1`; logged; rebuilt and hash-checked) |
| `external` | `ecgaug-v5-external`, `ecgaug-v1-checkpoints`, `ecgaug-p7-runs` | INCART and SVDB |

Settings: **Internet on**, Accelerator **None**. Download the zip in Output and run `python kaggle/ingest_runs.py <zip>`.


In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
PART = "validation"
DATABASE = "all"                      # external only: "incartdb", "svdb" or "all"
RUN_TESTS = False                     # the full test suite first; nothing else runs if it fails
INPUTS_DIR = "/kaggle/input"          # attached datasets: banks, checkpoints (runs/), external data (data/)
GIT_REF = "main"                      # branch name or full commit SHA to run
DEVICE = "auto"                       # "auto" uses the GPU when the session has one
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # attached mitdb-raw files; if none, downloaded from PhysioNet's AWS mirror (checksums checked)
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"


In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
configs = ["configs/phase6_patient_adaptation/patient_adaptation.yaml"]
extra = ["--part", PART, "--device", DEVICE] + (["--database", DATABASE] if PART == "external" else [])
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.experiments.phase6_patient_adaptation", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs, "--raw-dir", RAW_DATASET_DIR, "--artifacts-dir", INPUTS_DIR, "--data-dir", INPUTS_DIR]
if RUN_TESTS:
    cmd += ["--tests"]
cmd += ["--", *extra]
if PART == "test":
    env["ALLOW_TEST_POSTHOC"] = "1"   # the one posthoc DS2 read of PREREGISTRATION v6; it is logged in docs/TEST_ACCESS_LOG.md
code = subprocess.run(cmd, cwd=REPO, env=env).returncode
print("finished with exit code", code, "(2 = the run was refused, read the message above)")
